In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings("ignore")


In [2]:
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")

features = ["Nombre de Titres", "Echéance", "Taux"]
target = "Montant"

X = df[features]
y = df[target]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)


In [4]:
def evaluate_model(name, model):
    results = {}

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Sélection de toutes les features
    selector = SelectKBest(score_func=f_regression, k="all")
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)

    # Grilles élargies
    param_grid = {}
    if name == "Ridge Regression":
        param_grid = {"alpha": np.linspace(0.0001, 10, 50)}
    elif name == "Lasso Regression":
        param_grid = {"alpha": np.linspace(0.0001, 1, 50)}
    elif name == "Decision Tree":
        param_grid = {
            "max_depth": [None, 5, 10, 20, 50],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    elif name == "Random Forest":
        param_grid = {
            "n_estimators": [100, 300, 500],
            "max_depth": [None, 10, 20, 50],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    elif name == "Gradient Boosting":
        param_grid = {
            "n_estimators": [100, 300, 500],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 5, 10],
            "subsample": [0.8, 1.0]
        }

    # Tuning si grille disponible
    if param_grid:
        grid = GridSearchCV(
            model, param_grid,
            cv=3,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )
        grid.fit(X_train_sel, y_train)
        best_model = grid.best_estimator_
        preds = best_model.predict(X_test_sel)
        results["RMSE tuning"] = rmse(y_test, preds)
    else:
        model.fit(X_train_sel, y_train)
        preds = model.predict(X_test_sel)
        results["RMSE tuning"] = rmse(y_test, preds)

    return results


In [5]:
models = {
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Linear Regression": LinearRegression()
}


In [8]:
results_all = {}

for name, model in models.items():
    results_all[name] = evaluate_model(name, model)

results_df = pd.DataFrame(results_all).T
results_df


NameError: name 'rmse' is not defined